# Sparky chatbot — live chat with SynapseGPT on Colab

Serves `sparky_chatbot.py` (Flask web UI, streaming) from a Colab GPU and
exposes it through an ngrok public URL you can open on any device.

**Needs a GPU runtime** (Runtime → Change runtime type → GPU; T4 works for
inference, A100 is snappier). Run the cells top to bottom; the last cell
stays "running" while the server is up — that's normal. Stop it to shut down.

In [ ]:
# 1) Mount Drive (checkpoint + tokenizer live there)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
SYNAPSE_DIR = '/content/drive/MyDrive/synapse'
print('SYNAPSE_DIR =', SYNAPSE_DIR)

In [ ]:
# 2) Clone or update the repo
import os, subprocess
REPO_DIR = '/content/synapse_repo'
REPO_URL = 'https://github.com/ajencinas/synapse.git'
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('repo exists — pulling latest')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR], check=True)
assert os.path.isfile(os.path.join(REPO_DIR, 'sparky', 'sparky_chatbot.py'))
print('REPO_DIR =', REPO_DIR)

In [ ]:
# 3) GPU check + deps (torch is preinstalled on Colab)
import torch
assert torch.cuda.is_available(), 'no GPU — Runtime → Change runtime type → GPU'
print('GPU:', torch.cuda.get_device_name(0))
!pip -q install flask tokenizers

In [ ]:
# 4) Pick the checkpoint and copy it to fast local disk
#    'sft'      -> sft_checkpoints/v2_12source/sft_best.pth   (archived v2 ship model)
#    'pretrain' -> checkpoints/synapse_2b_d2560_l28.pth (base model)
VARIANT = 'sft'

import os, shutil
DRIVE_CKPT = {
    'sft':      os.path.join(SYNAPSE_DIR, 'sft_checkpoints', 'v3_15source', 'sft_best.pth'),
    'pretrain': os.path.join(SYNAPSE_DIR, 'checkpoints', 'synapse_2b_d2560_l28.pth'),
}[VARIANT]
DRIVE_TOK = os.path.join(SYNAPSE_DIR, 'tokenizer_out', 'tokenizer.json')
for p in (DRIVE_CKPT, DRIVE_TOK):
    assert os.path.exists(p), f'missing on Drive: {p} — train first / check the path'

CKPT_LOCAL = f'/content/{os.path.basename(DRIVE_CKPT)}'
TOK_LOCAL = '/content/tokenizer.json'
for src, dst in ((DRIVE_CKPT, CKPT_LOCAL), (DRIVE_TOK, TOK_LOCAL)):
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
        print(f'copying {src} -> {dst} ...')
        shutil.copyfile(src, dst)
print(f'ready: {CKPT_LOCAL} ({os.path.getsize(CKPT_LOCAL)/1e9:.2f} GB)')

In [ ]:
# 5) ngrok: install the binary and store the authtoken in sparky/.env
#    Token source: Drive file synapse/ngrok_token.txt if present, else a prompt.
#    (Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken)
import os
if not os.path.exists('/usr/local/bin/ngrok'):
    !curl -sL https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz \
      | tar xz -C /usr/local/bin
!ngrok version

token_file = os.path.join(SYNAPSE_DIR, 'ngrok_token.txt')
if os.path.exists(token_file):
    NGROK_TOKEN = open(token_file).read().strip()
    print('token loaded from Drive')
else:
    from getpass import getpass
    NGROK_TOKEN = getpass('paste your ngrok authtoken: ').strip()
    open(token_file, 'w').write(NGROK_TOKEN)  # saved to Drive for next time
assert NGROK_TOKEN, 'empty ngrok token'
with open(os.path.join(REPO_DIR, 'sparky', '.env'), 'w') as f:
    f.write(f'NGROK_AUTHTOKEN={NGROK_TOKEN}\n')
print('sparky/.env written')

# Brave key for the search tool (optional but recommended): Colab Secrets -> BRAVE_API_KEY
BRAVE = ''
try:
    from google.colab import userdata
    BRAVE = userdata.get('BRAVE_API_KEY') or ''
except Exception:
    pass
print('search tool:', 'ENABLED' if BRAVE else 'DISABLED (no BRAVE_API_KEY secret — python tool still works)')

In [ ]:
# 6) SAFETY: unmount Drive before serving. The tool loop executes model-written
#    python in a sandbox that blocks NETWORK, not the filesystem — nothing of yours
#    should be reachable from this VM while the chatbot is up. Everything needed
#    was copied to /content in step 4.
from google.colab import drive
drive.flush_and_unmount()
import os; assert not os.path.exists('/content/drive/MyDrive'), 'Drive still mounted'
print('Drive unmounted — safe to serve')

In [ ]:
# 7) Launch. The ngrok URL prints below — open it in any browser.
#    This cell keeps running while the server is up; stop it to shut down.
#    Settings ▸ 'Tools (python)' toggles the canonical tool prompt + sandbox.
env = f'BRAVE_API_KEY={BRAVE}' if BRAVE else ''
!cd {REPO_DIR}/sparky && {env} python sparky_chatbot.py --no-download \
    --ckpt {CKPT_LOCAL} --tokenizer {TOK_LOCAL}

## Notes
- The chatbot detects an SFT checkpoint (`stage=='sft'`) and uses the trained
  ChatML template. **Tools ON** sends `CANONICAL_TOOL_SYSTEM` and runs `<|tool_call|>`
  python in the sandbox (shown as red-bordered blocks); **Tools OFF** sends no system
  prompt at all — the two distributions the model was trained on, nothing else.
- Try both legs: *"What is 4913 × 27381?"* (should call python) and
  *"What is 6 times 7?"* or a prose question (should answer directly).
- v3 searches on factoid questions by itself; news/'search for X' phrasing triggers
  the chatbot's forced-search assist (\U0001F50D search \u00b7 auto block). Both need the
  BRAVE_API_KEY secret; without it the python tool still works.
- Re-mount Drive (cell 1) if you need to pull a different checkpoint.
- If `torch.compile` misbehaves on the Colab GPU, add `--no-compile` to the launch cell.